In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import pandas as pd
import numpy as np
import os
import shutil
import random
import time
from sklearn.metrics import accuracy_score, f1_score, classification_report
from collections import Counter
import json

# =====================================================================
# 1. SETUP & PATH CONFIGURATION (KAGGLE PATHS)
# =====================================================================
# Converted from the original Colab version, which mounted Google Drive
# and unzipped images into a local folder each run. This mirrors the
# same Kaggle input/output layout used by the SSL fine-tuning notebook
# so both scripts read the same dataset split and write comparable
# output formats.

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

BACKUP_DIR = '/kaggle/input/datasets/ittisamurtunib/dataset/CSVs-20260711T054424Z-2-001/CSVs'
UNLABELED_DIR = '/kaggle/input/datasets/ittisamurtunib/dataset/ISIC_2019_Training_Input/ISIC_2019_Training_Input'

OUTPUT_DIR = '/kaggle/working/Thesis_Outputs'
LABELED_DIR = '/kaggle/working/data/labeled_real'

os.makedirs(LABELED_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(UNLABELED_DIR):
    raise FileNotFoundError(f"Could not locate image directory at {UNLABELED_DIR}")

## LOAD DATA
train_df = pd.read_csv(os.path.join(BACKUP_DIR, 'expA_train.csv'))
val_df = pd.read_csv(os.path.join(BACKUP_DIR, 'expA_val.csv'))
test_df = pd.read_csv(os.path.join(BACKUP_DIR, 'expA_test.csv'))
all_images = pd.concat([train_df, val_df, test_df])['image'].unique()

## Copy the 691 labeled images into the writable working directory
found = 0
for img_id in all_images:
    src = f'{UNLABELED_DIR}/{img_id}.jpg'
    dst = f'{LABELED_DIR}/{img_id}.jpg'
    if os.path.exists(src):
        if not os.path.exists(dst):
            shutil.copy(src, dst)
        found += 1
print(f"Copied/verified {found}/{len(all_images)} labeled images into working memory.")

IMAGE_DIR = LABELED_DIR

print("Class distribution (train):")
print(train_df['label'].value_counts())

# =====================================================================
# 2. AUGMENTATION & DATASET
# =====================================================================

normal_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

mel_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(45),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.85, 1.15)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class AugmentedDataset(Dataset):
    def __init__(self, df, image_dir, mel_multiplier=5, transform_normal=None, transform_mel=None):
        self.df = df
        self.image_dir = image_dir
        self.classes = sorted(df['label'].unique())
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.transform_normal = transform_normal
        self.transform_mel = transform_mel

        mel_rows = df[df['label'] == 'MEL']
        other_rows = df[df['label'] != 'MEL']
        mel_expanded = pd.concat([mel_rows] * mel_multiplier, ignore_index=True)
        self.expanded_df = pd.concat([mel_expanded, other_rows], ignore_index=True)\
                              .sample(frac=1, random_state=42).reset_index(drop=True)

        print(f"Original: {len(df)} | Expanded: {len(self.expanded_df)}")
        print(f"Expanded distribution: {self.expanded_df['label'].value_counts().to_dict()}")

    def __len__(self):
        return len(self.expanded_df)

    def __getitem__(self, idx):
        row = self.expanded_df.iloc[idx]
        img_path = f"{self.image_dir}/{row['image']}.jpg"
        img = Image.open(img_path).convert('RGB')
        label = self.class_to_idx[row['label']]

        if row['label'] == 'MEL' and self.transform_mel:
            img = self.transform_mel(img)
        elif self.transform_normal:
            img = self.transform_normal(img)
        else:
            img = test_transform(img)

        return img, label

# =====================================================================
# 3. RESNET18 MODEL
# =====================================================================

class ResNetClassifier(nn.Module):
    def __init__(self, num_classes, freeze_early=True):
        super().__init__()
        resnet = models.resnet18(pretrained=True)
        self.encoder = nn.Sequential(*list(resnet.children())[:-1])

        if freeze_early:
            for param in list(self.encoder.parameters())[:30]:  # Approx first 6 blocks
                param.requires_grad = False

        self.classifier = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        features = self.encoder(x)
        features = features.squeeze(-1).squeeze(-1)
        return self.classifier(features)

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=1.5):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        return ((1 - pt) ** self.gamma * ce_loss).mean()

# =====================================================================
# 4. TRAIN / EVAL HELPERS
# =====================================================================
#
# CHECKPOINT-SELECTION FIX (same rationale as the SSL fine-tuning script):
# checkpoints are now selected by validation MACRO-F1 rather than plain
# validation accuracy, so the ResNet-18 baseline is judged by the same
# melanoma-sensitive criterion as the proposed method -- an apples-to-
# apples comparison instead of one model being early-stopped on a
# different metric than the other.

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct = 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            preds = outputs.argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return acc, macro_f1, all_labels, all_preds

def measure_latency(model, loader, n_batches=10):
    """Average per-image inference time (ms), measured on GPU if available."""
    model.eval()
    times = []
    with torch.no_grad():
        for i, (images, _) in enumerate(loader):
            if i >= n_batches:
                break
            images = images.to(device)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            start = time.perf_counter()
            _ = model(images)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            elapsed = time.perf_counter() - start
            times.append(elapsed / images.size(0))
    return float(np.mean(times) * 1000) if times else None  # ms per image

# =====================================================================
# 5. 3-SEED ROBUSTNESS RUN
# =====================================================================

seeds = [42, 123, 2024]
all_results = {}
param_counts_recorded = False

for seed in seeds:
    print(f"\n{'#'*40}")
    print(f"SEED {seed}")
    print(f"{'#'*40}")

    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    train_ds = AugmentedDataset(train_df, IMAGE_DIR, mel_multiplier=5,
                                 transform_normal=normal_transform, transform_mel=mel_transform)
    train_ds.expanded_df = train_ds.expanded_df.sample(frac=1, random_state=seed).reset_index(drop=True)
    val_ds = AugmentedDataset(val_df, IMAGE_DIR, mel_multiplier=1,
                               transform_normal=test_transform, transform_mel=test_transform)
    test_ds = AugmentedDataset(test_df, IMAGE_DIR, mel_multiplier=1,
                                transform_normal=test_transform, transform_mel=test_transform)

    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=16)
    test_loader = DataLoader(test_ds, batch_size=16)

    model = ResNetClassifier(len(train_ds.classes), freeze_early=True).to(device)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    if not param_counts_recorded:
        print(f"Total params: {total_params:,} | Trainable: {trainable_params:,}")
        param_counts_recorded = True

    weight_dict = {'MEL': 4.0, 'BKL': 2.0, 'NV': 1.0}
    weights_list = [weight_dict[cls] for cls in train_ds.classes]
    weights = torch.tensor(weights_list, dtype=torch.float32).to(device)
    criterion = FocalLoss(alpha=weights, gamma=1.5)
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0001)

    epochs = 30
    best_val_macro_f1 = -1.0
    patience = 7
    epochs_no_improve = 0
    history = {"train_loss": [], "train_acc": [], "val_acc": [], "val_macro_f1": []}
    seed_ckpt = f'{OUTPUT_DIR}/best_resnet18_seed{seed}.pth'
    best_epoch = -1

    for epoch in range(epochs):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
        val_acc, val_macro_f1, _, _ = evaluate(model, val_loader)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["val_macro_f1"].append(val_macro_f1)

        if (epoch + 1) % 3 == 0:
            print(f"Epoch {epoch+1}: Train={train_acc:.3f}, Val Acc={val_acc:.3f}, "
                  f"Val Macro-F1={val_macro_f1:.3f}")

        if val_macro_f1 > best_val_macro_f1:
            best_val_macro_f1 = val_macro_f1
            best_epoch = epoch + 1
            torch.save(model.state_dict(), seed_ckpt)
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

    print(f"Best checkpoint: epoch {best_epoch} (Val Macro-F1={best_val_macro_f1:.3f})")

    model.load_state_dict(torch.load(seed_ckpt, map_location=device))
    test_acc, test_macro_f1, true_labels, pred_labels = evaluate(model, test_loader)

    report = classification_report(true_labels, pred_labels,
                                    target_names=train_ds.classes, output_dict=True)
    mel_recall = report['MEL']['recall']
    latency_ms = measure_latency(model, test_loader)

    print(f"\n>>> Seed {seed} RESULTS: Test Acc={test_acc:.3f}, "
          f"Test Macro-F1={test_macro_f1:.3f}, MEL Recall={mel_recall:.1%}, "
          f"Latency={latency_ms:.2f} ms/image <<<")
    print(f"Predictions: {Counter(pred_labels)}")

    all_results[seed] = {
        "model": "ResNet18_pretrained",
        "augmentation": "heavy_mel_5x",
        "weights": weight_dict,
        "test_accuracy": test_acc,
        "test_macro_f1": test_macro_f1,
        "mel_recall": mel_recall,
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_val_macro_f1,
        "total_params": total_params,
        "trainable_params": trainable_params,
        "latency_ms_per_image": latency_ms,
        "report": report,
        "history": history,
        "checkpoint_selection_metric": "val_macro_f1",
    }
    torch.save(model.state_dict(), f'{OUTPUT_DIR}/resnet18_seed{seed}_finetuned.pth')

# =====================================================================
# 3-SEED SUMMARY (same format as the SSL + rebalance script, for direct comparison)
# =====================================================================
accs = [all_results[s]["test_accuracy"] for s in seeds]
f1s = [all_results[s]["test_macro_f1"] for s in seeds]
recalls = [all_results[s]["mel_recall"] for s in seeds]
latencies = [all_results[s]["latency_ms_per_image"] for s in seeds]

print("\n" + "="*50)
print("3-SEED SUMMARY -- ResNet18 + Heavy Augmentation (checkpoint by val macro-F1)")
print("="*50)
for s in seeds:
    print(f"Seed {s}: Test Acc={all_results[s]['test_accuracy']:.3f}, "
          f"Test Macro-F1={all_results[s]['test_macro_f1']:.3f}, "
          f"MEL Recall={all_results[s]['mel_recall']:.1%}, "
          f"Best epoch={all_results[s]['best_epoch']}")
print(f"\nMean Test Accuracy: {np.mean(accs):.3f} +/- {np.std(accs):.3f}")
print(f"Mean Test Macro-F1: {np.mean(f1s):.3f} +/- {np.std(f1s):.3f}")
print(f"Mean MEL Recall:    {np.mean(recalls):.1%} +/- {np.std(recalls):.1%}")
print(f"Mean Latency:       {np.mean(latencies):.2f} +/- {np.std(latencies):.2f} ms/image")
print(f"Total params:       {all_results[seeds[0]]['total_params']:,} "
      f"(trainable: {all_results[seeds[0]]['trainable_params']:,})")

with open(f'{OUTPUT_DIR}/resnet18_3seed_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print(f"\nSaved full 3-seed results to {OUTPUT_DIR}/resnet18_3seed_results.json")


Device: cuda
Copied/verified 691/691 labeled images into working memory.
Class distribution (train):
label
NV     349
BKL     99
MEL     35
Name: count, dtype: int64

########################################
SEED 42
########################################
Original: 483 | Expanded: 623
Expanded distribution: {'NV': 349, 'MEL': 175, 'BKL': 99}
Original: 104 | Expanded: 104
Expanded distribution: {'NV': 76, 'BKL': 21, 'MEL': 7}
Original: 104 | Expanded: 104
Expanded distribution: {'NV': 75, 'BKL': 21, 'MEL': 8}


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 183MB/s]


Total params: 11,242,563 | Trainable: 10,559,491
Epoch 3: Train=0.860, Val Acc=0.798, Val Macro-F1=0.512
Epoch 6: Train=0.926, Val Acc=0.712, Val Macro-F1=0.464
Epoch 9: Train=0.952, Val Acc=0.769, Val Macro-F1=0.484
Epoch 12: Train=0.970, Val Acc=0.779, Val Macro-F1=0.540
Early stopping at epoch 12
Best checkpoint: epoch 5 (Val Macro-F1=0.541)

>>> Seed 42 RESULTS: Test Acc=0.798, Test Macro-F1=0.603, MEL Recall=37.5%, Latency=1.28 ms/image <<<
Predictions: Counter({np.int64(2): 76, np.int64(0): 17, np.int64(1): 11})

########################################
SEED 123
########################################
Original: 483 | Expanded: 623
Expanded distribution: {'NV': 349, 'MEL': 175, 'BKL': 99}
Original: 104 | Expanded: 104
Expanded distribution: {'NV': 76, 'BKL': 21, 'MEL': 7}
Original: 104 | Expanded: 104
Expanded distribution: {'NV': 75, 'BKL': 21, 'MEL': 8}


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 3: Train=0.867, Val Acc=0.721, Val Macro-F1=0.456
Epoch 6: Train=0.925, Val Acc=0.692, Val Macro-F1=0.502
Epoch 9: Train=0.970, Val Acc=0.808, Val Macro-F1=0.532
Epoch 12: Train=0.971, Val Acc=0.788, Val Macro-F1=0.496
Epoch 15: Train=0.973, Val Acc=0.769, Val Macro-F1=0.490
Epoch 18: Train=0.970, Val Acc=0.750, Val Macro-F1=0.501
Epoch 21: Train=0.986, Val Acc=0.683, Val Macro-F1=0.421
Epoch 24: Train=0.995, Val Acc=0.702, Val Macro-F1=0.425
Early stopping at epoch 24
Best checkpoint: epoch 17 (Val Macro-F1=0.553)

>>> Seed 123 RESULTS: Test Acc=0.808, Test Macro-F1=0.598, MEL Recall=25.0%, Latency=1.27 ms/image <<<
Predictions: Counter({np.int64(2): 78, np.int64(0): 21, np.int64(1): 5})

########################################
SEED 2024
########################################
Original: 483 | Expanded: 623
Expanded distribution: {'NV': 349, 'MEL': 175, 'BKL': 99}
Original: 104 | Expanded: 104
Expanded distribution: {'NV': 76, 'BKL': 21, 'MEL': 7}
Original: 104 | Expanded: 104


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 3: Train=0.849, Val Acc=0.673, Val Macro-F1=0.523
Epoch 6: Train=0.942, Val Acc=0.596, Val Macro-F1=0.441
Epoch 9: Train=0.971, Val Acc=0.692, Val Macro-F1=0.460
Epoch 12: Train=0.957, Val Acc=0.721, Val Macro-F1=0.483
Epoch 15: Train=0.978, Val Acc=0.702, Val Macro-F1=0.468
Early stopping at epoch 17
Best checkpoint: epoch 10 (Val Macro-F1=0.598)

>>> Seed 2024 RESULTS: Test Acc=0.750, Test Macro-F1=0.573, MEL Recall=37.5%, Latency=1.28 ms/image <<<
Predictions: Counter({np.int64(2): 69, np.int64(0): 22, np.int64(1): 13})

3-SEED SUMMARY -- ResNet18 + Heavy Augmentation (checkpoint by val macro-F1)
Seed 42: Test Acc=0.798, Test Macro-F1=0.603, MEL Recall=37.5%, Best epoch=5
Seed 123: Test Acc=0.808, Test Macro-F1=0.598, MEL Recall=25.0%, Best epoch=17
Seed 2024: Test Acc=0.750, Test Macro-F1=0.573, MEL Recall=37.5%, Best epoch=10

Mean Test Accuracy: 0.785 +/- 0.025
Mean Test Macro-F1: 0.591 +/- 0.013
Mean MEL Recall:    33.3% +/- 5.9%
Mean Latency:       1.28 +/- 0.00 ms/image


In [9]:
import os

print("=== Contents of /kaggle/input ===")
print(os.listdir('/kaggle/input'))

print("\n=== All subfolders inside /kaggle/input ===")
for root, dirs, files in os.walk('/kaggle/input'):
    print(root)
    if len(files) > 0:
        print(f"   Files: {files[:3]}...") # prints first 3 files

=== Contents of /kaggle/input ===
['datasets']

=== All subfolders inside /kaggle/input ===
/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/ittisamurtunib
/kaggle/input/datasets/ittisamurtunib/dataset
/kaggle/input/datasets/ittisamurtunib/dataset/CSVs-20260711T054424Z-2-001
/kaggle/input/datasets/ittisamurtunib/dataset/CSVs-20260711T054424Z-2-001/CSVs
   Files: ['dataset_config_summary.txt', 'expC_test.csv', 'expB_val.csv']...
/kaggle/input/datasets/ittisamurtunib/dataset/ISIC_2019_Training_Input
/kaggle/input/datasets/ittisamurtunib/dataset/ISIC_2019_Training_Input/ISIC_2019_Training_Input
   Files: ['ISIC_0057312.jpg', 'ISIC_0014233_downsampled.jpg', 'ISIC_0059626.jpg']...
